In [25]:
import torch
import torchvision
import torchvision.models as models
from torchvision.transforms import v2 as transforms
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image

In [15]:
class SiameseResNet50(nn.Module):    
    def __init__(self, embedding_dim=128, freeze_backbone=False):
        super(SiameseResNet50, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)        
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        feature_dim = 2048
        self.embedding_head = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embedding_dim)
        )
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
    
    def forward(self, x):
        features = self.backbone(x)
        features = features.view(features.size(0), -1)
        
        embeddings = self.embedding_head(features)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return embeddings


In [16]:
class ContrastiveLoss(nn.Module):
    
    def __init__(self, margin=1.0, temperature=0.07):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
        self.temperature = temperature
    
    def forward(self, embeddings1, embeddings2, labels):

        distances = torch.norm(embeddings1 - embeddings2, dim=1)
        loss = labels * distances.pow(2) + \
               (1 - labels) * F.relu(self.margin - distances).pow(2)
        
        return loss.mean()

In [27]:
class SiameseDataset(Dataset):
    
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform
        
        if self.transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToImage(),
                transforms.ToDtype(torch.float32, scale=True),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                )
            ])
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img1_path = row['img1']
        img2_path = row['img2']
        label = float(row['label'])
        
        try:
            img1 = Image.open(img1_path).convert('RGB')
            img2 = Image.open(img2_path).convert('RGB')
        except Exception as e:
            print(f"Error loading images at index {idx}: {e}")
            raise
        
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        
        return img1, img2, torch.tensor(label, dtype=torch.float32)

In [21]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    
    for batch_idx, (img1, img2, labels) in enumerate(train_loader):
        img1, img2 = img1.to(device), img2.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()        
        emb1 = model(img1)
        emb2 = model(img2)
        
        loss = criterion(emb1, emb2, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    return avg_loss

In [22]:
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for img1, img2, labels in val_loader:
            img1, img2 = img1.to(device), img2.to(device)
            labels = labels.to(device)
            
            emb1 = model(img1)
            emb2 = model(img2)
            
            loss = criterion(emb1, emb2, labels)
            total_loss += loss.item()
    
    avg_loss = total_loss / len(val_loader)
    return avg_loss

In [30]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.cuda.empty_cache()
    print(f"Using device: {device}")
    
    model = SiameseResNet50(embedding_dim=128, freeze_backbone=False)
    model = model.to(device)
    
    criterion = ContrastiveLoss(margin=1.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    input_file_path = "pair_input.csv"
    
    dataset = SiameseDataset(input_file_path)
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
    
    num_epochs = 5
    print("Starting training...")
    
    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = validate(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        scheduler.step()
    
    print("Training complete!")
    
    torch.save(model.state_dict(), "siamese_resnet50.pth")

Using device: cuda
Starting training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 3.63 GiB of which 21.88 MiB is free. Including non-PyTorch memory, this process has 3.60 GiB memory in use. Of the allocated memory 3.39 GiB is allocated by PyTorch, and 140.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)